# SimpleModel workflow

This notebook shows the new config-first `SimpleModel` workflow:

1. Create or load a grid
2. Build a `SimpleModelConfig`
3. Create a model with `build_simple_model(...)`
4. Run MODFLOW 6
5. Inspect the results

The example below uses a tiny hand-built Voronoi grid so the setup stays focused on the model workflow itself.

In [ ]:
from pathlib import Path

import numpy as np
import simple_modflow as mf

In [ ]:
# A tiny 2-cell Voronoi grid with clockwise polygons.
verts = np.array(
    [
        [0.0, 0.0],
        [1.0, 0.0],
        [1.0, 1.0],
        [0.0, 1.0],
        [2.0, 0.0],
        [2.0, 1.0],
    ],
    dtype=float,
)
iverts = [[0, 3, 2, 1], [1, 2, 5, 4]]
xcyc = np.array([[0.5, 0.5], [1.5, 0.5]], dtype=float)

vor = mf.VoronoiGridPlus(verts=verts, iverts=iverts, xcyc=xcyc)
vor.gdf_vorPolys

In [ ]:
# Keep the model name at 16 characters or fewer because MF6 enforces that limit.
workspace = Path.cwd().resolve().parents[1] / "artifacts" / "simple_model_workflow"
workspace.mkdir(parents=True, exist_ok=True)

config = mf.SimpleModelConfig(
    vor=vor,
    name="smpl_demo",
    mf_folder_path=workspace,
    nper=1,
    nlay=1,
    grid_type="disv",
    top=[10.0, 9.0],
    bottom=[[0.0, 0.0]],
    initial_heads=[10.0, 9.0],
    k=[1.0, 1.0],
    save_specific_discharge=False,
    boundary_mode="chd",
    boundary_cells=[0, 1],
    boundary_head=[10.0, 9.0],
    sto_steady={0: True},
    sto_transient={},
)
config

In [ ]:
model = mf.build_simple_model(config)
success, output = model.run_simulation()
success

In [ ]:
heads = model.hds.get_data(kstpkper=(0, 0)).squeeze()
heads

In [ ]:
model.all_heads[["elev", "geometry"]]

## Where to go from here

- Replace the hand-built `VoronoiGridPlus` with a grid built from `TriangleGrid`
- Swap `boundary_mode="chd"` for `"drain"` or `None`
- Add `rch_dict` for recharge-driven examples
- Use the returned `SimulationBase` object for heads, budgets, plots, and post-processing